# 4. The live SPHEREx archive path

How CHRONOS reads real SPHEREx Quick Release files from IRSA. The first cells build one file in the documented layout and read it with the same code. The last cell asks the real archive; it reports when the archive cannot be reached.

In [1]:
import os, sys, json, math, warnings
warnings.filterwarnings("ignore")
ROOT = os.path.abspath("..")
sys.path.insert(0, os.path.join(ROOT, "backend"))
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
for f in os.listdir(os.path.join(ROOT, "scripts", "fonts")):
    font_manager.fontManager.addfont(os.path.join(ROOT, "scripts", "fonts", f))
plt.rcParams.update({"font.family": "EB Garamond", "font.size": 12, "figure.facecolor": "#f5f0e6", "axes.facecolor": "#fbf8f2"})
import core, sky, pipeline as P
import io
from astropy.io import fits
import connectors as cn

## One file in the IRSA layout

Extensions IMAGE, FLAGS (bit names in MP keywords), VARIANCE, ZODI, and the alternate WCS A for detector pixels.

In [2]:
spec = sky.make_spec(3)
E, truth, cat = sky.render(spec)
e = E[0]
hd = e.wcs.to_header()
hd["DETECTOR"], hd["OBSID"], hd["VERSION"] = e.det, e.id, "6.5.6"
hd["CRPIX1A"], hd["CRPIX2A"], hd["CRVAL1A"], hd["CRVAL2A"] = 1.0, 1.0, e.dxy[0], e.dxy[1]
fh = fits.Header()
for name, bit in core.FLAG_BITS.items():
    fh["MP_" + name[:5]] = bit
f = fits.HDUList([fits.PrimaryHDU(), fits.ImageHDU(e.img.astype("f4"), hd, name="IMAGE"), fits.ImageHDU(e.flags.astype("i4"), fh, name="FLAGS"),
                  fits.ImageHDU(e.var.astype("f4"), name="VARIANCE"), fits.ImageHDU(np.full(e.img.shape, e.zodi, "f4"), name="ZODI")])
f.info()
print("flag names read from the file:", cn.flag_bits(f))

Filename: (No file associated with this HDUList)
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       4   ()      
  1  IMAGE         1 ImageHDU        34   (49, 49)   float32   
  2  FLAGS         1 ImageHDU        24   (49, 49)   int32   
  3  VARIANCE      1 ImageHDU         8   (49, 49)   float32   
  4  ZODI          1 ImageHDU         8   (49, 49)   float32   


flag names read from the file: {'TRANS': 0, 'OVERF': 1, 'SUR_E': 2, 'PHANT': 4, 'REFER': 5, 'NONFU': 6, 'DICHR': 7, 'MISSI': 9, 'HOT': 10, 'COLD': 11, 'FULLS': 12, 'PHANM': 14, 'NONLI': 15, 'PERSI': 17, 'OUTLI': 19, 'SOURC': 21}


## The search and cutout requests that live mode sends

In [3]:
ra, dec = 269.44850, 4.73942
print(cn.SIA + "?" + "&".join(f"{k}={v}" for k, v in dict(COLLECTION="spherex_qr3", POS=f"CIRCLE {ra} {dec} 0.12", MAXREC=2000).items()))
print("cutout: <access_url>?center=%.5f,%.5f&size=0.08" % (ra, dec))

https://irsa.ipac.caltech.edu/SIA?COLLECTION=spherex_qr3&POS=CIRCLE 269.4485 4.73942 0.12&MAXREC=2000
cutout: <access_url>?center=269.44850,4.73942&size=0.08


## Ask the real archive

This cell needs internet access to irsa.ipac.caltech.edu.

In [4]:
try:
    rows = cn.search(ra, dec, 0.05, "spherex_qr3")
    print(len(rows), "SPHEREx QR3 files cover Barnard's Star")
    for r in rows[:5]:
        print(r.get("obs_id"), r.get("energy_bandpassname"), r.get("t_min"))
except Exception as err:
    print("The archive could not be reached from this machine:", type(err).__name__, str(err)[:120])
    print("Run this notebook with internet access, or start the app and press Check the live archive.")

The archive could not be reached from this machine: ProxyError HTTPSConnectionPool(host='irsa.ipac.caltech.edu', port=443): Max retries exceeded with url: /SIA?COLLECTION=spherex_qr3&
Run this notebook with internet access, or start the app and press Check the live archive.
